In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, TensorDataset 

from sklearn.metrics import roc_auc_score
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler

import scipy
import warnings

import xgboost

In [2]:
device = torch.device('cuda') if torch.cuda.is_available else torch.device('cpu')
np.random.seed(42)
warnings.simplefilter('ignore')

In [3]:
train = pd.read_csv('./data/train.csv')
val = pd.read_csv('./data/test.csv')

In [4]:
train.sample(5)

,id,Driver,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,PitNextLap
157757,157757,D072,HARD,Miami Grand Prix,2023,0,3,1,3.0,11,94.500,-1.341,-10.570,0.052632,0.0,0.0
212674,212674,D211,HARD,United States Grand Prix,2023,0,46,2,25.0,10,102.939,0.344,-20.719,0.821429,0.0,0.0
208745,208745,D304,HARD,Miami Grand Prix,2023,0,14,2,7.0,19,93.746,0.077,-17.333,0.250000,1.0,0.0
350823,350823,D033,HARD,Azerbaijan Grand Prix,2025,0,36,3,19.0,13,107.800,23.846,-8.905,0.507042,4.0,1.0
3349,3349,FRE,SOFT,Bahrain Grand Prix,2022,0,3,1,3.0,13,101.173,-1.845,25.964,0.042857,-5.0,0.0


In [5]:
val.sample(5)

,id,Driver,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change
86604,525744,SPE,HARD,Canadian Grand Prix,2025,0,60,3,30.0,8,77.323,-4.085,-1.813,0.789474,8.0
854,439994,CEL,HARD,Mexico City Grand Prix,2024,0,49,2,20.0,2,80.930,-17.929,-39.979,0.628205,-1.0
171974,611114,TRU,MEDIUM,Canadian Grand Prix,2023,0,3,1,3.0,5,77.229,-0.467,-3.849,0.042857,0.0
28435,467575,D037,HARD,Italian Grand Prix,2024,0,19,2,10.0,13,86.159,7.033,-44.322,0.267606,4.0
175988,615128,BRA,MEDIUM,Italian Grand Prix,2025,0,17,1,17.0,17,84.738,2.985,-139.233,0.223684,-1.0


In [6]:
train.describe()

,id,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,PitNextLap
count,439140.000000,439140.000000,439140.000000,439140.000000,439140.000000,439140.000000,439140.000000,439140.000000,439140.000000,439140.000000,439140.000000,439140.000000,439140.000000
mean,219569.500000,2023.523544,0.136118,23.105909,1.789113,14.158231,9.630339,90.948735,-3.770040,-25.721759,0.337661,0.101542,0.198982
std,126768.942943,1.024930,0.342915,16.958261,0.950194,9.801338,5.278770,19.772769,43.945759,54.766573,0.253277,4.006765,0.399235
min,0.000000,2022.000000,0.000000,1.000000,1.000000,1.000000,1.000000,67.694000,-2403.895000,-274.564000,0.012821,-18.000000,0.000000
25%,109784.750000,2023.000000,0.000000,9.000000,1.000000,6.000000,5.000000,82.621000,-8.884000,-46.566250,0.129870,-1.000000,0.000000
50%,219569.500000,2024.000000,0.000000,19.000000,2.000000,12.000000,10.000000,90.521000,-0.295000,-20.994000,0.269231,0.000000,0.000000
75%,329354.250000,2024.000000,0.000000,36.000000,2.000000,20.000000,14.000000,98.471000,0.115000,-6.199000,0.513158,2.000000,0.000000
max,439139.000000,2025.000000,1.000000,78.000000,8.000000,77.000000,20.000000,2507.607000,2423.932000,2412.026000,1.000000,18.000000,1.000000


In [7]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 439140 entries, 0 to 439139
Data columns (total 16 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   id                      439140 non-null  int64  
 1   Driver                  439140 non-null  object 
 2   Compound                439140 non-null  object 
 3   Race                    439140 non-null  object 
 4   Year                    439140 non-null  int64  
 5   PitStop                 439140 non-null  int64  
 6   LapNumber               439140 non-null  int64  
 7   Stint                   439140 non-null  int64  
 8   TyreLife                439140 non-null  float64
 9   Position                439140 non-null  int64  
 10  LapTime (s)             439140 non-null  float64
 11  LapTime_Delta           439140 non-null  float64
 12  Cumulative_Degradation  439140 non-null  float64
 13  RaceProgress            439140 non-null  float64
 14  Position_Change     

In [8]:
train['Year'] = train['Year'].astype('object')
val['Year'] = val['Year'].astype('object')

In [9]:
train['Driver'].nunique()

887

In [10]:
train_b_trans = train.drop(columns=['id', 'PitNextLap', 'Driver'])
test = train['PitNextLap']

In [11]:
val_b_trans = val.drop(['id'], axis=1)

In [12]:
number_columns_name = train_b_trans.select_dtypes(include='number').columns
cat_columns_name = train_b_trans.select_dtypes(include='object').columns

In [13]:
cat_columns_name

Index(['Compound', 'Race', 'Year'], dtype='object')

In [14]:
preprocessing = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), number_columns_name),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_columns_name)
    ]
)

In [15]:
df_train = preprocessing.fit_transform(train_b_trans)
df_train_processed = pd.DataFrame(df_train, columns=preprocessing.get_feature_names_out())

df_val = preprocessing.fit_transform(val_b_trans)
df_val_processed = pd.DataFrame(df_val, columns=preprocessing.get_feature_names_out())

In [16]:
X_train, X_test, y_train, y_test = train_test_split(
    df_train_processed, test, test_size=0.3, random_state=42
)

In [17]:
training_set = DataLoader(
    TensorDataset(
        torch.tensor(np.array(X_train), dtype=torch.float32), 
        torch.tensor(y_train, dtype=torch.long),
        ), batch_size=64, shuffle=True)

testing_set = DataLoader(
    TensorDataset(
        torch.tensor(np.array(X_test), dtype=torch.float32), 
        torch.tensor(np.array(y_test), dtype=torch.long),
        ), batch_size=64, shuffle=False)

In [18]:
class ANN(nn.Module):
    def __init__(self, input_size = X_train.shape[1]):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input_size, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(64, 2),
        )

    def forward(self, x):
        x = self.model(x)
        return x


In [19]:
X_train.shape[1]

45

In [20]:
model = ANN()
model.to(device)

optimizer = optim.Adam(model.parameters(), lr = 0.01)
critrion = nn.CrossEntropyLoss()
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', patience=5)

In [21]:
# Learning
best_para = 0.0
for epoch in range(30):
    model.train()
    running_loss = 0.0
    for inputs, labels in training_set:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = critrion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
    
    # average learning loss per 1 epoch
    train_loss = running_loss / len(training_set)
    
    # Validation
    model.eval()
    val_running_loss = 0.0
    val_probs = []
    val_targets = []
    
    with torch.no_grad():
        for inputs, labels in testing_set:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            
            # Validation Loss
            v_loss = critrion(outputs, labels)
            val_running_loss += v_loss.item()
            
            # probability culculation for AUC
            probs = torch.softmax(outputs, dim=1)[:, 1]
            val_probs.extend(probs.cpu().numpy())
            val_targets.extend(labels.cpu().numpy())
    
    avg_val_loss = val_running_loss / len(testing_set)
    auc = roc_auc_score(val_targets, val_probs)
    scheduler.step(auc)
    
    print(f"Epoch {epoch+1:02d}: "
          f"Train Loss: {train_loss:.4f} | "
          f"Val Loss: {avg_val_loss:.4f} | "
          f"Val AUC: {auc:.4f}")
    if best_para < auc:
        best_para = auc
        best_weights = model.state_dict()

Epoch 01: Train Loss: 0.2863 | Val Loss: 0.2622 | Val AUC: 0.9288
Epoch 02: Train Loss: 0.2747 | Val Loss: 0.2545 | Val AUC: 0.9352
Epoch 03: Train Loss: 0.2708 | Val Loss: 0.2511 | Val AUC: 0.9354
Epoch 04: Train Loss: 0.2673 | Val Loss: 0.2496 | Val AUC: 0.9368
Epoch 05: Train Loss: 0.2660 | Val Loss: 0.2489 | Val AUC: 0.9386
Epoch 06: Train Loss: 0.2647 | Val Loss: 0.2524 | Val AUC: 0.9377
Epoch 07: Train Loss: 0.2635 | Val Loss: 0.2454 | Val AUC: 0.9392
Epoch 08: Train Loss: 0.2624 | Val Loss: 0.2446 | Val AUC: 0.9395
Epoch 09: Train Loss: 0.2615 | Val Loss: 0.2464 | Val AUC: 0.9388
Epoch 10: Train Loss: 0.2619 | Val Loss: 0.2584 | Val AUC: 0.9348
Epoch 11: Train Loss: 0.2604 | Val Loss: 0.2428 | Val AUC: 0.9411
Epoch 12: Train Loss: 0.2606 | Val Loss: 0.2422 | Val AUC: 0.9406
Epoch 13: Train Loss: 0.2596 | Val Loss: 0.2439 | Val AUC: 0.9405
Epoch 14: Train Loss: 0.2588 | Val Loss: 0.2413 | Val AUC: 0.9417
Epoch 15: Train Loss: 0.2596 | Val Loss: 0.2464 | Val AUC: 0.9391
Epoch 16: 

In [22]:
val_set = DataLoader(
    TensorDataset(
        torch.tensor(np.array(df_val_processed), dtype=torch.float32)
        ), batch_size=64, shuffle=False)


In [23]:
model.load_state_dict(best_weights)
model.eval()
test_probs = []

with torch.no_grad():
    for (inputs,) in val_set:
        outputs = model(inputs.to(device))
        probs = torch.softmax(outputs, dim=1)[:, 1] # probability of class 1
        test_probs.extend(probs.cpu().numpy())

# save submission file
submission = pd.DataFrame({
    'id': val['id'],
    'PitNextLap': test_probs
})

submission.to_csv('output3.csv', index=False)

In [24]:
len(val_probs)

131742

In [25]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score

In [26]:
xgb_model = XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric='auc',
    random_state=42
)

cat_model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    eval_metric='AUC',
    random_seed=42,
    verbose=False
)

In [27]:
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)

xgb_val_probs   = xgb_model.predict_proba(df_val_processed)[:, 1]
xgb_test_probs  = xgb_model.predict_proba(X_test)[:, 1]
xgb_auc         = roc_auc_score(y_test, xgb_test_probs)
print(f"XGBoost  AUC: {xgb_auc:.4f}")

XGBoost  AUC: 0.9485


In [28]:
cat_model.fit(
    X_train, y_train,
    eval_set=(X_test, y_test)
)

cat_val_probs   = cat_model.predict_proba(df_val_processed)[:, 1]
cat_test_probs  = cat_model.predict_proba(X_test)[:, 1]
cat_auc         = roc_auc_score(y_test, cat_test_probs)
print(f"CatBoost AUC: {cat_auc:.4f}")

CatBoost AUC: 0.9438


In [29]:
ann_auc = roc_auc_score(y_test, val_probs)
print(f"ANN      AUC: {ann_auc:.4f}")

# ── 4. Export individual CSVs ─────────────────────────────────────────────────
pd.DataFrame({'id': val['id'], 'PitNextLap': xgb_val_probs})\
    .to_csv('submission_xgb.csv', index=False)

pd.DataFrame({'id': val['id'], 'PitNextLap': cat_val_probs})\
    .to_csv('submission_cat.csv', index=False)

ANN      AUC: 0.9425


In [30]:
results = {
    'ANN':      (ann_auc,  test_probs),       # your existing val probs
    'XGBoost':  (xgb_auc,  xgb_val_probs),
    'CatBoost': (cat_auc,  cat_val_probs),
}

print("\n── Model Comparison ──")
for name, (auc, _) in sorted(results.items(), key=lambda x: -x[1][0]):
    print(f"  {name:<10} AUC: {auc:.4f}")

best_name, (best_auc, best_probs) = max(results.items(), key=lambda x: x[1][0])
print(f"\n✓ Best model: {best_name}  (AUC: {best_auc:.4f})")

pd.DataFrame({'id': val['id'], 'PitNextLap': best_probs})\
    .to_csv('submission_best.csv', index=False)

print(f"  Saved → submission_best.csv")

# ── 6. Optional: ensemble all three (average probabilities) ───────────────────
ensemble_probs = (np.array(test_probs) + xgb_val_probs + cat_val_probs) / 3
pd.DataFrame({'id': val['id'], 'PitNextLap': ensemble_probs})\
    .to_csv('submission_ensemble.csv', index=False)

ensemble_test  = (np.array(val_probs) + xgb_test_probs + cat_test_probs) / 3
ensemble_auc   = roc_auc_score(y_test, ensemble_test)
print(f"\nEnsemble AUC: {ensemble_auc:.4f}  → submission_ensemble.csv")


── Model Comparison ──
  XGBoost    AUC: 0.9485
  CatBoost   AUC: 0.9438
  ANN        AUC: 0.9425

✓ Best model: XGBoost  (AUC: 0.9485)
  Saved → submission_best.csv

Ensemble AUC: 0.9475  → submission_ensemble.csv


In [31]:
xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)

xgb_val_probs   = xgb_model.predict_proba(df_val_processed)[:, 1]
xgb_test_probs  = xgb_model.predict_proba(X_test)[:, 1]
xgb_auc         = roc_auc_score(y_test, xgb_test_probs)
print(f"XGBoost  AUC: {xgb_auc:.4f}")

XGBoost  AUC: 0.9485


In [32]:
xgb_output = xgb_model.predict(df_val_processed)

In [33]:
from lightgbm import LGBMClassifier
from lightgbm import LGBMClassifier, early_stopping
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score

In [34]:
lgb_model = LGBMClassifier()

In [35]:
from lightgbm import LGBMClassifier

lgb_model = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    num_leaves=31,          # key LGB param: controls tree complexity
    subsample=0.8,          # row sampling per tree
    colsample_bytree=0.8,   # feature sampling per tree
    min_child_samples=20,   # min data in a leaf, prevents overfitting
    reg_alpha=0.1,          # L1 regularization
    reg_lambda=0.1,         # L2 regularization
    class_weight='balanced', # handles class imbalance automatically
    random_state=42,
    n_jobs=-1,
    verbose=-1              # suppress output
)

lgb_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    
)

,boosting_type,'gbdt'
,num_leaves,31
,max_depth,6
,learning_rate,0.05
,n_estimators,500
,subsample_for_bin,200000
,objective,None
,class_weight,'balanced'
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [36]:


# ── LightGBM ──────────────────────────────────────────────────────────────────
lgb_model = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_samples=20,
    reg_alpha=0.1,
    reg_lambda=0.1,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

lgb_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    callbacks=[early_stopping(stopping_rounds=50)]
)

lgb_val_probs  = lgb_model.predict_proba(df_val_processed)[:, 1]
lgb_test_probs = lgb_model.predict_proba(X_test)[:, 1]
lgb_auc        = roc_auc_score(y_test, lgb_test_probs)
print(f"LightGBM AUC: {lgb_auc:.4f}")

Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[500]	valid_0's binary_logloss: 0.301989
LightGBM AUC: 0.9466


## hare


In [37]:
lgb_test_probs = lgb_model.predict_proba(df_val_processed)[:, 1]

In [38]:

pd.DataFrame({'id': val['id'], 'PitNextLap': lgb_test_probs}).to_csv('submission_lgb.csv', index=False)

In [39]:
xgb_model = XGBClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.1,
    eval_metric='auc',
    early_stopping_rounds=50,
    random_state=42,
    n_jobs=-1,
    verbosity=0
)

xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)

xgb_val_probs  = xgb_model.predict_proba(df_val_processed)[:, 1]
xgb_test_probs = xgb_model.predict_proba(X_test)[:, 1]
xgb_auc        = roc_auc_score(y_test, xgb_test_probs)
print(f"XGBoost  AUC: {xgb_auc:.4f}")

XGBoost  AUC: 0.9485


In [40]:
cat_model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    eval_metric='AUC',
    early_stopping_rounds=50,
    random_seed=42,
    verbose=False
)

cat_model.fit(
    X_train, y_train,
    eval_set=(X_test, y_test)
)

cat_val_probs  = cat_model.predict_proba(df_val_processed)[:, 1]
cat_test_probs = cat_model.predict_proba(X_test)[:, 1]
cat_auc        = roc_auc_score(y_test, cat_test_probs)
print(f"CatBoost AUC: {cat_auc:.4f}")

CatBoost AUC: 0.9438


In [41]:
# ANN inference on X_test
model.load_state_dict(best_weights)
model.eval()
ann_test_probs = []

with torch.no_grad():
    for Xb, _ in testing_set:
        output = model(Xb.to(device))
        probs  = torch.softmax(output, dim=1)[:, 1]
        ann_test_probs.extend(probs.cpu().numpy())

In [42]:
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score

# ── 1. Load all submissions ───────────────────────────────────────────────────
submissions = {
    'LGB':      pd.read_csv('Output/submission_lgb.csv'),
    'XGBoost':  pd.read_csv('Output/submission_xgb.csv'),
    'CatBoost': pd.read_csv('Output/submission_cat.csv'),
    'Ensemble': pd.read_csv('Output/submission_ensemble.csv'),
    'Best':     pd.read_csv('Output/submission_best.csv'),
    'sample':   pd.read_csv('Output/output3.csv'),
}

# ── 2. Score each against y_test ─────────────────────────────────────────────
print("── Model AUCs ──")
all_preds = np.column_stack([df['PitNextLap'].values for df in submissions.values()])
final_preds = all_preds.mean(axis=1)

# ── 3. Save ───────────────────────────────────────────────────────────────────
pd.DataFrame({'id': val['id'], 'PitNextLap': final_preds}).to_csv('final_submission.csv', index=False)
print("Saved → final_submission.csv")

── Model AUCs ──
Saved → final_submission.csv


In [ ]:
from sklearn.ensemble import (
    BaggingClassifier, VotingClassifier,
    RandomForestClassifier, ExtraTreesClassifier,
    GradientBoostingClassifier, AdaBoostClassifier
)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
import numpy as np
import pandas as pd

# ── 1. Define all models ──────────────────────────────────────────────────────

# Bagged tree models (from before)
bagging_lgb = BaggingClassifier(
    estimator=LGBMClassifier(n_estimators=300, learning_rate=0.05, num_leaves=31, verbose=-1, random_state=42),
    n_estimators=5, max_samples=0.8, max_features=0.8, bootstrap=True, random_state=42, n_jobs=-1
)

bagging_xgb = BaggingClassifier(
    estimator=XGBClassifier(n_estimators=300, learning_rate=0.05, max_depth=6, verbosity=0, random_state=42),
    n_estimators=5, max_samples=0.8, max_features=0.8, bootstrap=True, random_state=42, n_jobs=-1
)

bagging_cat = BaggingClassifier(
    estimator=CatBoostClassifier(iterations=300, learning_rate=0.05, depth=6, verbose=False, random_seed=42),
    n_estimators=5, max_samples=0.8, max_features=0.8, bootstrap=True, random_state=42, n_jobs=-1
)

# Random Forest
rf_model = RandomForestClassifier(
    n_estimators=500,
    max_depth=8,
    min_samples_leaf=20,
    max_features='sqrt',
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

# Extra Trees
et_model = ExtraTreesClassifier(
    n_estimators=500,
    max_depth=8,
    min_samples_leaf=20,
    max_features='sqrt',
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

# Gradient Boosting
gb_model = GradientBoostingClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    random_state=42
)

# AdaBoost
ada_model = AdaBoostClassifier(
    n_estimators=200,
    learning_rate=0.05,
    random_state=42
)

# Logistic Regression
lr_model = LogisticRegression(
    C=0.1,
    class_weight='balanced',
    max_iter=1000,
    random_state=42,
    n_jobs=-1
)

# SVM (probability=True needed for predict_proba)
# svm_model = SVC(
#     C=1.0,
#     kernel='rbf',
#     probability=True,
#     class_weight='balanced',
#     random_state=42
# )

# KNN
knn_model = KNeighborsClassifier(
    n_neighbors=15,
    weights='distance',
    metric='euclidean',
    n_jobs=-1
)

# MLP (sklearn neural net)
mlp_model = MLPClassifier(
    hidden_layer_sizes=(128, 64),
    activation='relu',
    learning_rate_init=0.001,
    max_iter=200,
    early_stopping=True,
    random_state=42
)

# ── 2. Fit all models ─────────────────────────────────────────────────────────
models = {
    'Bag-LGB':  bagging_lgb,
    'Bag-XGB':  bagging_xgb,
    'Bag-CAT':  bagging_cat,
    'RF':       rf_model,
    'ExtraTrees': et_model,
    'GBM':      gb_model,
    'AdaBoost': ada_model,
    'LR':       lr_model,
    'KNN':      knn_model,
    'MLP':      mlp_model,
}

for name, model in models.items():
    print(f"Fitting {name}...")
    model.fit(X_train, y_train)
print("All models fitted ✓")

# ── 3. Soft Voting across all ─────────────────────────────────────────────────
voting = VotingClassifier(
    estimators=[(name, model) for name, model in models.items()],
    voting='soft',
    n_jobs=-1
)

print("\nFitting Voting Classifier...")
voting.fit(X_train, y_train)

# ── 4. Predict on val_set ─────────────────────────────────────────────────────
print("\n── Predicting ──")
all_val_preds = {}

for name, model in models.items():
    all_val_preds[name] = model.predict_proba(df_val_processed)[:, 1]
    print(f"  {name} ✓")

all_val_preds['Voting'] = voting.predict_proba(df_val_processed)[:, 1]

# ── 5. Export individual CSVs ─────────────────────────────────────────────────
print("\n── Saving submissions ──")
for name, preds in all_val_preds.items():
    fname = f"submission_{name.lower().replace('-', '_')}.csv"
    pd.DataFrame({'id': val['id'], 'PitNextLap': preds}).to_csv(fname, index=False)
    print(f"  Saved → {fname}")

# ── 6. Load previous submissions & blend everything ───────────────────────────
prev = {
    'LGB':      pd.read_csv('submission_lgb.csv')['PitNextLap'].values,
    'XGBoost' : pd.read_csv('submission_xgb.csv')['PitNextLap'].values,
    'CatBoost': pd.read_csv('submission_cat.csv')['PitNextLap'].values,
    'Ensemble': pd.read_csv('submission_ensemble.csv')['PitNextLap'].values
}

all_preds = np.column_stack(
    list(prev.values()) +
    list(all_val_preds.values())
)

final_preds = all_preds.mean(axis=1)



Fitting Bag-LGB...
Fitting Bag-XGB...
Fitting Bag-CAT...
Fitting RF...
Fitting ExtraTrees...
Fitting GBM...
Fitting AdaBoost...
Fitting LR...
Fitting KNN...
Fitting MLP...
All models fitted ✓

Fitting Voting Classifier...

── Predicting ──
  Bag-LGB ✓
  Bag-XGB ✓
  Bag-CAT ✓
  RF ✓
  ExtraTrees ✓
  GBM ✓
  AdaBoost ✓
  LR ✓
  KNN ✓
  MLP ✓

── Saving submissions ──
  Saved → submission_bag_lgb.csv
  Saved → submission_bag_xgb.csv
  Saved → submission_bag_cat.csv
  Saved → submission_rf.csv
  Saved → submission_extratrees.csv
  Saved → submission_gbm.csv
  Saved → submission_adaboost.csv
  Saved → submission_lr.csv
  Saved → submission_knn.csv
  Saved → submission_mlp.csv
  Saved → submission_voting.csv

✓ Saved → final_submission.csv  (blend of 15 models)


In [46]:
pd.DataFrame({'id': val['id'],'PitNextLap': final_preds}).to_csv('final_submission12.csv', index=False)
